In [0]:
%pip install --upgrade --quiet langchain==0.1.16 langchain-core langchain_community==0.0.36 langchain-experimental youtube_search wikipedia==1.4.0 duckduckgo-search
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from langchain_community.chat_models import ChatDatabricks

llm_dbrx = ChatDatabricks(
    endpoint="databricks-dbrx-instruct",
    max_tokens=500
)

In [0]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import YouTubeSearchTool
from langchain.agents import Tool
from langchain_experimental.utilities import PythonREPL
from langchain_community.tools import DuckDuckGoSearchRun

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=100)
tool_wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

tool_youtube = YouTubeSearchTool()

search = DuckDuckGoSearchRun()

python_repl = PythonREPL()

repl_tool = Tool(name="python_repl", description="A Python Shell", func=python_repl.run)

tools = [tool_wiki, tool_youtube, search, repl_tool]


In [0]:
from langchain.prompts import PromptTemplate

template = '''
Answer the following questions as best as you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
'''

prompt = PromptTemplate.from_template(template)

In [0]:
from langchain.agents import AgentExecutor
from langchain.agents.react.agent import create_react_agent

agent = create_react_agent(llm_dbrx, tools, prompt=prompt)
brixo = AgentExecutor(agent=agent, tools=tools, verbose=True)
brixo.invoke({"input":
    """
    Which movie depicts the WWI's western front? Follow the steps below:
    1. decide which movies depict the WWI's western front
    2. show the trailer of the movies that has the best ratings or randomly recommend one
    3. collect data about the movie using the search tool, draw a bar chart using the Python libraries. If you can't find latest data use any dummy data that you can make up.  Don't use ```for Python code. Any leading or trailing backticks should be removed from the input. If the input starts with Python, remove that word as well (for both lower and upper case). The output must be the result of executed code.
    4. identify yourself as an AI Agent.
    """
    })



> Entering new AgentExecutor chain...


---------------------------------------------------------------------------
HTTPError                                 Traceback (most recent call last)
File /databricks/python/lib/python3.12/site-packages/mlflow/utils/request_utils.py:63, in augmented_raise_for_status(response)
     62 try:
---> 63     response.raise_for_status()
     64 except HTTPError as e:

File /databricks/python/lib/python3.12/site-packages/requests/models.py:1024, in Response.raise_for_status(self)
   1023 if http_error_msg:
-> 1024     raise HTTPError(http_error_msg, response=self)

HTTPError: 404 Client Error: Failed to find MT LLM Endpoint for requested endpoint: databricks-dbrx-instruct for url: https://dbc-c38475c8-7939.cloud.databricks.com/serving-endpoints/databricks-dbrx-instruct/invocations

During handling of the above exception, another exception occurred:

HTTPError                                 Traceback (most recent call last)
File <command-7835425163815266>, line 6
      4 agent = create_react_a